In [23]:
from unstructured.partition.pdf import partition_pdf
from pdf2image import convert_from_path
import os

In [24]:
dpi_setting=300

In [25]:
pdf_path=r'luso_bank_2025-06-30.pdf'
pages_to_test=(3,3)
images = convert_from_path(
        pdf_path, 
        first_page=pages_to_test[0], 
        last_page=pages_to_test[1],
        dpi=dpi_setting # 200 DPI 對 AI 嚟講已經足夠睇清楚無線表格
    )

In [26]:
for i, image in enumerate(images):
    temp_name = f"temp_page_{pages_to_test[0] + i}.png"
    image.save(temp_name, "PNG")
    print(f"已生成測試圖：{temp_name}")


已生成測試圖：temp_page_3.png


In [27]:
import shutil

# 1. 自動搵出 tesseract 嘅路徑
tesseract_path = shutil.which("tesseract")

if tesseract_path:
    # 2. 將 tesseract 所在嘅資料夾加入 PATH
    tesseract_dir = os.path.dirname(tesseract_path)
    if tesseract_dir not in os.environ["PATH"]:
        os.environ["PATH"] += os.pathsep + tesseract_dir
    print(f"✅ 搵到 Tesseract: {tesseract_path}")
else:
    # 如果仲係搵唔到，手動指去 Homebrew 預設路徑
    os.environ["PATH"] += os.pathsep + "/opt/homebrew/bin"
    print("⚠️ 未能自動識別，已嘗試手動加入 Homebrew 預設路徑")

✅ 搵到 Tesseract: /opt/homebrew/bin/tesseract


In [28]:
from PIL import Image, ImageOps

def preprocess_image(img_path):
    img = Image.open(img_path).convert('L') # 轉灰色
    # 增加對比度，令字體更黑，背景更白
    img = ImageOps.autocontrast(img)
    # 二值化處理 (門檻可以根據情況調整)
    img = img.point(lambda x: 0 if x < 140 else 255, '1')
    img.save("processed_page.png")

preprocess_image("temp_page_3.png")

In [29]:
from unstructured.partition.image import partition_image

elements = partition_image(
    # filename=f"temp_page_{pages_to_test[0]}.png",
    filename='processed_page.png',
    strategy="hi_res",
    infer_table_structure=True,
    # 🌟 官方建議改用 languages
    languages=["chi_tra", "eng"],  
    # 🌟 選用對中文佈局更敏感嘅模型
    hi_res_model_name="yolox",
    # 🌟 強制提高 OCR 的解析品質
    pdf_image_dpi=dpi_setting,
    # chunking_strategy="by_title",
    extra_configs=['--psm 6', '--oem 3']
)

In [30]:

import pandas as pd
from io import StringIO

# 1. 準備一個 List 裝起所有搵到嘅 DataFrame
all_tables = []

for el in elements:
    # 2. 淨係要 Table 類別
    if el.category == "Table":
        # unstructured 已經幫你將無線表格轉好咗做 HTML 原始碼
        html_str = el.metadata.text_as_html
        
        if html_str:
            # 3. 用返我哋之前學過嘅 StringIO 轉做 DataFrame
            df = pd.read_html(StringIO(html_str))[0]
            all_tables.append(df)

# 4. 睇下第一張表係咪你要嗰張
if all_tables:
    print(f"✅ 成功提取 {len(all_tables)} 張表格！")
    print(all_tables[0].head())
else:
    print("❌ 雖然有 elements，但入面好似冇 Table 類別。")

✅ 成功提取 1 張表格！
           Unnamed: 0                                             ASSETS  \
0          現金 及 銀行 存款                       Cash and balances with banks   
1  存放 本 地 監管 機 羿 之 存款           Deposits with local regulatory authority   
2                 NaN         Deposits with the central bank in Mainland   
3                 NaN    China Placements with banks and other financial   
4                 NaN  institutions Financial assets purchased under ...   

   澳門 幣 千 元 MOP:000  澳門 幣 千 元 MOP'000  
0           9057025           8230695  
1           2605245           2350865  
2           2573124           8512698  
3           1198898            925633  
4           3150470           4717155  


In [31]:
all_tables[0]

,Unnamed: 0,ASSETS,澳門 幣 千 元 MOP:000,澳門 幣 千 元 MOP'000
0,現金 及 銀行 存款,Cash and balances with banks,9057025,8230695
1,存放 本 地 監管 機 羿 之 存款,Deposits with local regulatory authority,2605245,2350865
2,NaN,Deposits with the central bank in Mainland,2573124,8512698
3,NaN,China Placements with banks and other financial,1198898,925633
4,NaN,institutions Financial assets purchased under ...,3150470,4717155
5,NaN,Derivative financial assets,27541,140944
6,客戶 貸款 及 墊 款,Loans and advances to customers,107505747,101602214
7,俯 攤 餘 成 本 計量 的 債權 投資,Investments in debt instruments at amortised cost,25394741,36130016
8,俯 公 銜 價 值 計量 且 其 變動 計 人 其 也 全 面 收益 的 債權 投資,Debt instruments at fair value through other c...,46273235,34278139
9,俯 公 允 價 值 計量 且 其 變動 計 人 和 人 當 期 損益 的 金 融資 產,Financial assets at fair value through profit ...,3399188,793379
